In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyemu
import herebedragons as hbd
from pathlib import Path
from pyemu.emulators import DSI

# Data space inversion

One run of our model takes about 9 minutes. History matching it with an ensemble method means hundreds of runs per iteration, which is more compute than most of us have. So we build a fast stand-in for the model instead.

**Data space inversion** (DSI) learns the statistical relationship between the simulated observations and the forecasts from the prior Monte Carlo in `02_priormc`. Then it conditions the forecasts on the measured data directly, without running the model again. We won't go into the theory here; the GMDSI notebooks and videos cover it well. Here we show how to do it with `pyemu`.

What we need:
- the prior Monte Carlo results,
- some data to assimilate,
- a forecast we care about.

Everything in this notebook runs in a few minutes.

# Getting ready
## Load the prior Monte Carlo results

In [ ]:
m_d = Path("master_priormc")
pst = pyemu.Pst(str(m_d / "pest.pst"))
oe = pst.ies.obsen.copy()
oe.shape

## Choose the truth

We don't history match against the field data here. Instead we pick one prior realisation and pretend it is the truth. Its simulated values at the field sample times become our "measurements". Because we know the true forecast, we can check whether DSI gets it right.

We pick an inconvenient truth: a realisation whose peak sulfate in the blended pumped water is higher than in 90% of the prior.

In [ ]:
obs = pst.observation_data
fc_name = obs.loc[obs.obgnme=="forecast"].obsnme
fc_name = fc_name[fc_name.str.contains("blend")].values[0]

truth_index = hbd.choose_truth(oe, fc_name, q=0.9)

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
oe.loc[:, fc_name].hist(ax=ax, bins=30, color="0.5")
ax.axvline(oe.loc[truth_index, fc_name], color="r", label="truth")
ax.set_xlabel("peak SO4 in pumped water (mol/L)")
ax.legend()
fig.tight_layout()
truth_index, oe.loc[truth_index, fc_name]

We keep the truth aside and drop it from the training data.

In [ ]:
truth = oe.loc[truth_index]
data = oe.loc[~oe.index.isin([truth_index]), :]
assert data.shape[0] == oe.shape[0] - 1

## Set observation values and weights

The truth's simulated values become the observation values.

In [ ]:
obs.loc[truth.index, "obsval"] = truth.values
obs.loc[:, "weight"] = 0.0

We give weight only to observations that a field campaign could have collected: concentrations at the monitoring screens at the field sample times, during the history window. The history window is stress periods 0 to 20 (up to day 308); `wellopt` only starts pumping after it.

We use sulfate, oxygen, nitrate, pH and temperature. The standard deviations are rough measurement errors for each species. Temperature is stored as degrees Celsius divided by 1000, so 0.0005 is half a degree.

In [ ]:
stdev = {"so4": 5e-5, "o0": 3e-5, "no3": 3e-5, "ph": 0.1, "tmp": 5e-4}
t_hist = 308

sampled = hbd.sampled_obs(pst, m_d)
nz = obs.loc[sampled]
nz = nz.loc[nz.variable.isin(stdev.keys()) & (nz.time.astype(float) <= t_hist)]

obs.loc[nz.index, "standard_deviation"] = nz.variable.map(stdev)
obs.loc[nz.index, "weight"] = 1.0 / obs.loc[nz.index, "standard_deviation"]
nz.groupby("variable").size()

The pumping rates of `wellopt` are decision variables, and the truth has its own random rates. The forecast depends on them, so we tell DSI which rates the truth used by giving the pumped-rate observations a small standard deviation. This way we compare forecasts for the same pumping.

In [ ]:
dvo = obs.loc[obs.obgnme=="decvar_obs"].index
obs.loc[dvo, "standard_deviation"] = 1.0 # m3/d
obs.loc[dvo, "weight"] = 1.0
obs.loc[dvo, ["obsval", "weight"]]

# Build the DSI emulator

DSI only needs the observations with weight, the forecasts, and anything else we want to plot. We also keep the weekly sulfate at the `wellopt` screens for a time-series plot.

In [ ]:
welopt_so4 = obs.loc[obs.obsid.isin(["welopt-ly1", "welopt-ly3", "welopt-ly5"]) & (obs.variable=="so4")].index
keep_obs = obs.loc[obs.weight > 0].index.tolist()
keep_obs += obs.loc[obs.obgnme=="forecast"].index.tolist()
keep_obs += [o for o in welopt_so4 if o not in keep_obs]
len(keep_obs)

We use a normal score transform, which makes the data look more Gaussian before DSI fits them (usually a good idea).

In [ ]:
transforms = [{"type": "normal_score"}]

dsi = DSI(pst=pst,
          data=data[keep_obs],
          transforms=transforms,
          energy_threshold=0.9999, # the truncated-SVD energy threshold
          verbose=False)
dsi.fit()

The DSI "model" is a matrix multiplication: it takes a vector of DSI parameters and returns the simulated observations. That means we can pass a whole ensemble of parameter vectors at once. Let's try five random ones, with the last set to zero (the prior mean):

In [ ]:
pvals = pd.DataFrame(np.random.normal(0, 1, size=(5, dsi.s.shape[0])),
                     index=[f"real_{i}" for i in range(4)] + ["base"])
pvals.loc["base", :] = 0.0
dsi.predict(pvals).loc[:, fc_name]

# Set up PEST++

`prepare_pestpp` writes a PEST++ template for the emulator. With `use_runstor=True` the forward run uses the external run manager of `pestpp-ies`: PEST++ hands over the whole ensemble of parameters at once, and DSI evaluates them in one matrix multiplication instead of one run at a time.

In [ ]:
dsi_t_d = Path("pst_template_dsi")
dpst = dsi.prepare_pestpp(t_d=dsi_t_d, use_runstor=True)
hbd.get_bins(dsi_t_d)

DSI fits the data quickly, so three iterations are enough here. We use 1000 realisations, about as many as the training data.

In [ ]:
nreals = 1000
dpst.control_data.noptmax = 3
dpst.pestpp_options["ies_num_reals"] = nreals
dpst.write(str(dsi_t_d / "dsi.pst"), version=2)

# Run pestpp-ies with /e

We copy the template to a run folder and start `pestpp-ies` with `/e`, which switches on the external run manager. As far as PEST++ knows, this is a serial run.

In [ ]:
md_dsi = Path("master_dsi")
if md_dsi.exists():
    shutil.rmtree(md_dsi)
shutil.copytree(dsi_t_d, md_dsi)

pyemu.os_utils.run("pestpp-ies dsi.pst /e", cwd=md_dsi)

# Results

In [ ]:
dpst = pyemu.Pst(str(md_dsi / "dsi.pst"))
pd.DataFrame({"measured": dpst.ies.phimeas["mean"], "actual": dpst.ies.phiactual["mean"]})

Phi is the weighted misfit. The "measured" phi compares each realisation with its own noisy copy of the data. If the fit is as good as the measurement errors allow, it should be close to the number of weighted observations (1095). After three iterations it is about 1100, so we stop there. The "actual" phi compares with the noise-free truth and can drop lower.

The posterior observation ensemble is the last iteration's:

In [ ]:
obsen = dpst.ies.obsen
post = obsen.loc[dpst.control_data.noptmax]
prior = obsen.loc[0]
post.shape

How well do the posterior realisations fit the truth's data? One panel per species:

In [ ]:
dobs = dpst.observation_data
fig, axs = plt.subplots(1, len(stdev), figsize=(3 * len(stdev), 3))
for ax, var in zip(axs, stdev.keys()):
    names = [o for o in nz.index[nz.variable==var] if o in dobs.index]
    x = dobs.loc[names, "obsval"].values
    for i in post.index:
        ax.scatter(x, post.loc[i, names], c="b", marker=".", s=2, alpha=0.2)
    lims = [min(x.min(), post.loc[:, names].values.min()), max(x.max(), post.loc[:, names].values.max())]
    ax.plot(lims, lims, "r--")
    ax.set_title(var)
    ax.set_xlabel("truth")
axs[0].set_ylabel("posterior")
fig.tight_layout()

Now the forecast. Did assimilating the data narrow the range of peak sulfate, and does it still include the truth?

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
bins = np.linspace(data[fc_name].min(), data[fc_name].max(), 40)
data[fc_name].hist(ax=ax, bins=bins, color="0.5", alpha=0.5, label="prior (training data)")
post[fc_name].hist(ax=ax, bins=bins, color="b", alpha=0.5, label="posterior")
ax.axvline(truth[fc_name], color="r", label="truth")
ax.set_xlabel("peak SO4 in pumped water (mol/L)")
ax.legend()
fig.tight_layout()

pd.DataFrame({"prior": data[fc_name].quantile([0.05, 0.5, 0.95]),
              "posterior": post[fc_name].quantile([0.05, 0.5, 0.95])}).T

The data narrow the forecast a lot. The 5-95% range of peak sulfate shrinks from 6.7e-4 to 8.9e-4 mol/L in the prior to 7.6e-4 to 8.7e-4 mol/L in the posterior. The truth (8.6e-4 mol/L) is inside the posterior, but near its upper end.

And sulfate over time at the middle `wellopt` screen:

In [ ]:
so4_ly3 = obs.loc[welopt_so4]
so4_ly3 = so4_ly3.loc[so4_ly3.obsid=="welopt-ly3"].copy()
so4_ly3["t"] = so4_ly3.time.astype(float)
so4_ly3 = so4_ly3.sort_values("t")
names, t = so4_ly3.index, so4_ly3.t.values

fig, ax = plt.subplots(1, 1, figsize=(7, 3))
for i in data.index[:200]:
    ax.plot(t, data.loc[i, names], c="0.5", alpha=0.2, lw=0.5)
for i in post.index[:200]:
    ax.plot(t, post.loc[i, names], c="b", alpha=0.2, lw=0.5)
ax.plot(t, truth[names], c="r", lw=2, label="truth")
ax.axvline(t_hist, c="k", ls="--", lw=1)
ax.set_xlabel("time (days)")
ax.set_ylabel("SO4 (mol/L)")
ax.set_title("welopt-ly3: prior (grey), posterior (blue)")
ax.legend()
fig.tight_layout()

The timing is less good. There are no data at `wellopt`, so this is a pure prediction. Once the well starts pumping at day 308, the posterior jumps above the truth and stays above it until about day 450. Linear DSI has a hard time with sharp fronts like this one.

In `05_ies` we history match the full model against the same truth, and see how close the emulator came to the real thing.